In [18]:
qp = load_dataset("issai/kazqad-retrieval", "queries-and-passages")
print(qp)

DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'positive_passages', 'negative_passages'],
        num_rows: 3487
    })
    validation: Dataset({
        features: ['query_id', 'query', 'positive_passages', 'negative_passages'],
        num_rows: 548
    })
    test: Dataset({
        features: ['query_id', 'query', 'positive_passages', 'negative_passages'],
        num_rows: 1929
    })
})


In [19]:
split = qp["validation"]

queries = []
for row in split:
    relevant_ids = {p["docid"] for p in row["positive_passages"]}
    if not relevant_ids:
        continue
    queries.append({
        "query": row["query"],
        "relevant_docs": relevant_ids,
    })

print(f"Built {len(queries)} queries")
print(queries[0])

Built 548 queries
{'query': 'Энергетикалық алмасу дегеніміз не?', 'relevant_docs': {'518100_6_1'}}


In [20]:
MIN_CHARS = 20

train_corpus = corpus["train"]

chunks = []
for row in train_corpus:
    text = row["text"].strip()
    if len(text) < MIN_CHARS:
        continue
    chunks.append({"text": text, "doc_id": row["docid"]})

print(f"Kept {len(chunks)} / {len(train_corpus)} passages after filtering junk")
print(chunks[0])

Kept 806268 / 825309 passages after filtering junk
{'text': '| rowspan="" |2017\n|Жекпе-жек өнерінің сақшылары\n|"功守道"\n|монах \n| rowspan="" |2020\n|Мулан\n|"Mulan"\n|Император\n|}', 'doc_id': '137863_9_1'}


In [21]:
relevant_ids = set()
for q in queries:
    relevant_ids |= q["relevant_docs"]

print(f"{len(relevant_ids)} unique relevant passages across {len(queries)} queries")

import random
random.seed(42)
distractor_pool = [c for c in chunks if c["doc_id"] not in relevant_ids]
distractors = random.sample(distractor_pool, min(5000, len(distractor_pool)))

small_chunks = [c for c in chunks if c["doc_id"] in relevant_ids] + distractors
print(f"Small test corpus: {len(small_chunks)} passages")

702 unique relevant passages across 548 queries
Small test corpus: 5701 passages


In [ ]:
for name, cls in [("tfidf", TfidfRetriever), ("bm25", BM25Retriever), ("hybrid", HybridRetriever), ("dense", DenseRetriever)]:
    retriever = cls(small_chunks)
    results = evaluate_retriever(retriever, queries, top_k=5)
    print(name, results)    

tfidf {'precision@k': np.float64(0.13722627737226278), 'recall@k': np.float64(0.5875912408759124), 'mrr': np.float64(0.43430656934306566)}
bm25 {'precision@k': np.float64(0.12664233576642336), 'recall@k': np.float64(0.5383211678832117), 'mrr': np.float64(0.42840632603406326)}
hybrid {'precision@k': np.float64(0.14306569343065695), 'recall@k': np.float64(0.6021897810218978), 'mrr': np.float64(0.466970802919708)}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dense {'precision@k': np.float64(0.003284671532846715), 'recall@k': np.float64(0.016423357664233577), 'mrr': np.float64(0.007968369829683698)}


In [24]:
from sentence_transformers import SentenceTransformer
multilingual_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
retriever = DenseRetriever(small_chunks, model=multilingual_model)
results = evaluate_retriever(retriever, queries, top_k=5)
print("dense (multilingual)", results)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

dense (multilingual) {'precision@k': np.float64(0.04671532846715329), 'recall@k': np.float64(0.2116788321167883), 'mrr': np.float64(0.13336374695863745)}


In [25]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-large")
retriever = DenseRetriever(small_chunks, model=e5_model)
results = evaluate_retriever(retriever, queries, top_k=5)
print("dense (multilingual-e5-large, no prefix)", results)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

dense (multilingual-e5-large, no prefix) {'precision@k': np.float64(0.1875912408759124), 'recall@k': np.float64(0.7408759124087592), 'mrr': np.float64(0.6081204379562044)}


In [ ]:
random.seed(7) 
distractor_pool = [c for c in chunks if c["doc_id"] not in relevant_ids]
distractors2 = random.sample(distractor_pool, min(5000, len(distractor_pool)))
small_chunks2 = [c for c in chunks if c["doc_id"] in relevant_ids] + distractors2

retriever = DenseRetriever(small_chunks2, model=e5_model)
results2 = evaluate_retriever(retriever, queries, top_k=5)
print("dense (E5, seed=7 check)", results2)

dense (E5, seed=7 check) {'precision@k': np.float64(0.1875912408759124), 'recall@k': np.float64(0.7427007299270073), 'mrr': np.float64(0.6162712895377128)}


In [28]:
import csv
results_summary = [
       ("dense_english_only", 0.008, 0.016),
       ("dense_multilingual_minilm", 0.133, 0.212),
       ("bm25", 0.428, 0.538),
       ("tfidf", 0.434, 0.588),
       ("hybrid", 0.467, 0.602),
       ("dense_e5_seed42", 0.608, 0.741),
       ("dense_e5_seed7", 0.616, 0.743),
   ]
with open("kazqad_results_summary.csv", "w", newline="") as f:
       writer = csv.writer(f)
       writer.writerow(["method", "mrr", "recall_at_5"])
       writer.writerows(results_summary)